# 1.2 — Preprocesamiento y escalado a 256×256

Escala todas las imágenes de `data/interim/dataset_png/` a 256×256 píxeles
(resolución definida en `config['compressai']['training']['image_size']`).
Aplica sobre los tres datasets iterando sobre `config['external_datasets']`.
Muestra comparación visual antes/después para cada dataset.

**Autor:** Jorge Ceferino Valdez — Maestría en Informática y Sistemas (UNPA)

In [ ]:
import os
import sys
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

In [ ]:
project_root = Path(os.path.abspath('')).parent
if project_root not in sys.path:
    sys.path.insert(0, str(project_root))
print(f'project_root: {project_root}')

In [ ]:
from src.config import config, external_datasets, project_dir, interim_data_dir

In [ ]:
# Parámetros desde config
target_size = config['compressai']['training']['image_size']  # 256
print(f'Resolución objetivo: {target_size}x{target_size} px')

input_dir  = interim_data_dir() / 'dataset_png'
output_dir = interim_data_dir() / 'dataset_scaled'
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Entrada: {input_dir}')
print(f'Salida:  {output_dir}')

## 1. Funciones de redimensionado

Redimensiona preservando relación de aspecto con relleno negro centrado,
io directamente a la resolución objetivo sin padding (el modelo CompressAI
puede trabajar con imágenes cuadradas directas).

In [ ]:
def redimensionar_imagen(img: np.ndarray, target: int) -> np.ndarray:
    """Redimensiona preservando aspect ratio; rellena con negro hasta target×target."""
    h, w = img.shape[:2]
    scale = target / max(h, w)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    # Relleno centrado
    canvas = np.zeros((target, target, 3), dtype=np.uint8)
    y_off = (target - new_h) // 2
    x_off = (target - new_w) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = img_resized
    return canvas

def process_image(img_path: Path, output_subdir: Path, target: int) -> None:
    img = cv2.imread(str(img_path))
    if img is None:
        return
    img_scaled = redimensionar_imagen(img, target)
    output_subdir.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_subdir / img_path.name), img_scaled)

## 2. Escalado en paralelo

In [ ]:
# Construir lista de tareas (input_path, output_subdir)
tasks = []
for subdir, _, files in os.walk(input_dir):
    for file in files:
        if file.endswith('.png'):
            img_path = Path(subdir) / file
            rel = Path(subdir).relative_to(input_dir)
            out_sub = output_dir / rel
            tasks.append((img_path, out_sub))

print(f'Total imágenes a escalar: {len(tasks)}')

max_workers = None
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    executor.map(lambda t: process_image(t[0], t[1], target_size), tasks)

print(f'Escalado completo. Imágenes en: {output_dir}')

## 3. Verificación visual — comparación antes/después

In [ ]:
random.seed(42)

# Una imagen aleatoria por dataset de origen para la comparación visual
clases_dirs = [d for d in input_dir.iterdir() if d.is_dir()]
samples = random.sample(clases_dirs, min(4, len(clases_dirs)))

fig, axes = plt.subplots(len(samples), 2, figsize=(10, len(samples) * 3))
if len(samples) == 1:
    axes = [axes]

for i, clase_dir in enumerate(samples):
    imgs = list(clase_dir.glob('*.png'))
    if not imgs:
        continue
    img_path = random.choice(imgs)
    out_path = output_dir / clase_dir.name / img_path.name

    orig  = cv2.cvtColor(cv2.imread(str(img_path)),  cv2.COLOR_BGR2RGB)
    if out_path.exists():
        scaled = cv2.cvtColor(cv2.imread(str(out_path)), cv2.COLOR_BGR2RGB)
    else:
        scaled = np.zeros((target_size, target_size, 3), dtype=np.uint8)

    axes[i][0].imshow(orig)
    axes[i][0].set_title(f'Original {orig.shape[1]}x{orig.shape[0]}\n{clase_dir.name}', fontsize=9)
    axes[i][0].axis('off')

    axes[i][1].imshow(scaled)
    axes[i][1].set_title(f'Escalada {scaled.shape[1]}x{scaled.shape[0]}', fontsize=9)
    axes[i][1].axis('off')

plt.suptitle(f'Comparación antes/después — escalado a {target_size}x{target_size}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Verificación de distorsión

Verifica que el escalado no introduce distorsión relevante
comprobando que las imágenes escaladas tienen la resolución correcta.

In [ ]:
scaled_files = list(output_dir.rglob('*.png'))
print(f'Imágenes escaladas: {len(scaled_files)}')

errors = []
for f in random.sample(scaled_files, min(200, len(scaled_files))):
    img = cv2.imread(str(f))
    if img is None:
        errors.append(f)
        continue
    h, w = img.shape[:2]
    if h != target_size or w != target_size:
        errors.append(f)

if errors:
    print(f'ADVERTENCIA: {len(errors)} imágenes con resolución incorrecta:')
    for e in errors[:5]:
        img = cv2.imread(str(e))
        print(f'  {e.name}: {img.shape[1]}x{img.shape[0]}')
else:
    print(f'OK — todas las imágenes muestreadas son {target_size}x{target_size} px')

## 5. TODO — Dataset propio del dron

In [ ]:
# TODO: dataset propio del dron
drone_dest = project_dir() / external_datasets['drone_captures']['dest']

if drone_dest.exists() and any(drone_dest.rglob('*.jpg')):
    print('[drone_captures] Escalando imágenes del dron ...')
    drone_tasks = []
    for f in drone_dest.rglob('*.jpg'):
        out_sub = output_dir / 'drone_captures'
        drone_tasks.append((f, out_sub))
    for f in drone_dest.rglob('*.png'):
        out_sub = output_dir / 'drone_captures'
        drone_tasks.append((f, out_sub))
    with ThreadPoolExecutor(max_workers=None) as executor:
        executor.map(lambda t: process_image(t[0], t[1], target_size), drone_tasks)
    print(f'[drone_captures] {len(drone_tasks)} imágenes escaladas.')
else:
    print('[drone_captures] Carpeta no encontrada — pendiente (notebook 0.0).')

## 6. Verificación final

In [ ]:
print('=' * 55)
print('VERIFICACIÓN FINAL')
print('=' * 55)
assert output_dir.exists(), f'No existe: {output_dir}'
scaled_count = len(list(output_dir.rglob('*.png')))
assert scaled_count > 0, 'No hay imágenes escaladas'
print(f'  [OK] dataset_scaled: {scaled_count} imágenes en {output_dir}')
print()
print('Notebook 1.2 completado correctamente.')